In [1]:
from pathlib import Path

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.packaging.soma_calcium import package_soma_calcium_batch

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# -------------------------------------------------------------------
# 1) Build registry and collect candidate sessions
# -------------------------------------------------------------------

target_mice = [826033, 834788, 838410]

registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check","volume_imaging"],
    paradigms=["change_detection_passive"],
)

In [4]:
# -------------------------------------------------------------------
# 2) Select twelve sessions total
#    Here: earliest 6 from each mouse, or fewer if unavailable
# -------------------------------------------------------------------
process_df = process_df.sort_values(["subject_id", "session_date"])

selected_df = (
    process_df.groupby("subject_id", group_keys=False)
    .head(7)
    .reset_index(drop=True)
)

print(selected_df[["subject_id", "session_date", "session_type"]])
print(f"Selected {len(selected_df)} sessions total.")

    subject_id session_date session_type
0       826033   2026-02-21     familiar
1       826033   2026-02-23     familiar
2       826033   2026-02-24     familiar
3       826033   2026-02-25        novel
4       826033   2026-02-26       novel+
5       826033   2026-02-27       novel+
6       834788   2026-03-02     familiar
7       834788   2026-03-03     familiar
8       834788   2026-03-04     familiar
9       834788   2026-03-05     familiar
10      834788   2026-03-17        novel
11      834788   2026-03-19       novel+
12      834788   2026-03-20       novel+
13      838410   2026-03-02     familiar
14      838410   2026-03-03     familiar
15      838410   2026-03-04     familiar
16      838410   2026-03-05     familiar
17      838410   2026-03-18        novel
18      838410   2026-03-19       novel+
19      838410   2026-03-20       novel+
Selected 20 sessions total.


In [9]:
# -------------------------------------------------------------------
# 3) Resolve assets
# -------------------------------------------------------------------
assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

In [10]:
# -------------------------------------------------------------------
# 4) Choose output directory
# -------------------------------------------------------------------
output_root = Path(
    r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\analysis\derived\packaged\soma_calcium"
)

In [ ]:
# -------------------------------------------------------------------
# 5) Run packaging
# -------------------------------------------------------------------
results = package_soma_calcium_batch(
    assets=assets,
    output_root=output_root,
    trace_type="Fsvd",
    dmds=(1, 2),
    event_time_col="corrected_timestamps",
    overwrite=True,
    process_kwargs=dict(
        motion_correct=True,
        motion_regress_on="dF",
        motion_ridge=1e-2,
        motion_use_fields=(
            "onlineXshift",
            "onlineYshift",
            "onlineZshift",
            "motionDSr",
            "motionDSc",
        ),
        use_glu_as_motion_regressor=True,
        baseline_method="hull",
        denoise_window_s=2.0,
        hull_window_s=90.0,
        f0_floor_frac=0.05,
        mask_artifacts=True,
        std_factor=20.0,
        nan_pad=10,
        unmix=True,
        hp_window_s=0.5,
        ridge=1e-6,
    ),
)

In [ ]:
# -------------------------------------------------------------------
# 6) Inspect results
# -------------------------------------------------------------------
for r in results:
    print("=" * 80)
    print("session_id:", r.get("session_id"))
    print("output_dir :", r.get("output_dir"))
    print("status     :", r.get("status"))
    print("exported_dmds:", r.get("exported_dmds", []))
    if "errors" in r and r["errors"]:
        print("errors:")
        for err in r["errors"]:
            print("  -", err)